In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls "/content/drive/MyDrive"

 APP					   VID-20241014-WA0064.mp4
'Avinav Kumar Sansung Prismpdf.pdf'	   VID-20250101-WA0033.mp4
'Colab Notebooks'			   VID-20250101-WA0105.mp4
 Colab_Notebooks			   VID-20250103-WA0000.mp4
'ERP final report123 (2) (1).pdf'	   VID20250111172809.mp4
 IMG-20260403-WA0023.jpg		   VID20250609113557.mp4
'Presentation - ERP Implementation.pptx'   VID20250720185444.mp4
'SAP Course Completion BAdges'		   VID20250722152237.mp4
'SharpAiThon Participation.pdf'		   VID20250722162934.mp4
'SRM Project'				   VID20250722163057.mp4
 Untitled0.ipynb			   VID-20250810-WA0010.mp4
 VID-20230701-WA0051.mp4		   VID20251209123018.mp4
 VID20230809130045.mp4			   VID20251209123039.mp4
 VID20240619121622.mp4			  'VID-20260203-WA0001 (1).mp4'
 VID20240730205054.mp4			   VID-20260203-WA0001.mp4
 VID20240731135158.mp4			   VID-20260214-WA0012.mp4
 VID20240803182823.mp4			  'Videos of Phone'
 VID20240803182928.mp4			   warehouseImages1
 VID20240804094322.mp4			   WarehouseImages2
 VID20241014120329.mp4


In [4]:
import os

os.chdir("/content/drive/MyDrive/Colab_Notebooks")
print(os.getcwd())

/content/drive/MyDrive/Colab_Notebooks


In [5]:
!ls

F1_TextSummarizer.ipynb  samsum-test.csv   samsum-validation.csv
results			 samsum-train.csv


## We are going to use T5 model for this text summarizarion . go to Hugging face

In [6]:
!nvidia-smi # to check wheter google colab gpu is used or not

Sun Aug 23 07:55:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!pip install transformers
!pip install "transformers[torch]"

In [8]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments, T5ForConditionalGeneration

In [9]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [10]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [11]:
train_data.shape

(14732, 3)

In [12]:
# random sampling
train_data = train_data.sample(n=4000 , random_state=42).reset_index(drop=True) # first we are starting with 4k values later uf u have time do it on 14k data
val_data = val_data.sample(n=500 , random_state=42).reset_index(drop=True) # for validation we need less data

In [13]:
train_data.shape

(4000, 3)

## Data Preprocessing

In [14]:
import re # regularexpression
def clean_data(text):
    text = re.sub(r"\r\n"," ",text) #lines
    text = re.sub(r"\s+"," ",text) # spaces
    text = re.sub(r"<.*?>"," ",text) # html tags ,<p>,<h1>
    text = text.strip().lower()
    return text




In [15]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [16]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenization of data

In [17]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [18]:
#raw_data => tokenize inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"], padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"] # token_ids => add to the input as labels
    return inputs

In [19]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()


In [20]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [21]:
# input ids - dialogue => token ids
# 1->End of statement , 0 = padding
# attention mask (tells where is a valid token id and where is a padding using 1 and 0 in input_ids)
# labels - target => summary token


In [22]:
type(train_dataset)
type(val_dataset)


list

# Working with our Model

In [23]:
# NLP ->generation task
model= T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [24]:
# fine tunning
import torch

if torch.backends.mps.is_available():
    device =torch.device("mps")
elif torch.cuda.is_available():
    device= torch.device("cuda")
else:
    device =torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
# Training arguments
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs =6, # after the project u can increase it and train it for more epochs such as 10
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps = 500
    # 0-> lr default

)

In [26]:
trainer = Trainer(
    model =model,
    args=training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [30]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.340716,0.377756
2,0.394980,0.358170
3,0.372777,0.354403
4,0.361662,0.350815
5,0.354448,0.349556
6,0.351057,0.348913


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.5292734934488932, metrics={'train_runtime': 1307.505, 'train_samples_per_second': 18.356, 'train_steps_per_second': 2.294, 'total_flos': 3385710505623552.0, 'train_loss': 0.5292734934488932, 'epoch': 6.0})

In [ ]:
# with every epoch trained , it saves its states and parameters in the foilde resutls , that we can see
# now the imp. part is to save the model


In [31]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [34]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Test the core logic for summarization

In [35]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [36]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai technology continues to expand rapidly across industries, from healthcare to finance and education. ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
